# Võltsitud Pangakõned — Full Pipeline

Fake bank call fraud detection — end-to-end pipeline demonstration.

This notebook walks through every sprint of the project by importing the production functions from the project modules and showing intermediate results inline.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid')

print(f'Project root: {ROOT}')

---
# Sprint 1 — Data Preparation

## 1.1 Generate synthetic dataset

We generate 5000 fake bank calls with intentionally embedded fraud patterns: mass campaigns (one caller, many victims), temporal clustering, post-call behavior signals, and complaint keywords.

In [ ]:
from dataset.generate import generate_dataset

raw_df = generate_dataset(n_rows=5000, fraud_rate=0.15, seed=42)

print(f'Generated: {len(raw_df)} rows (5000 + injected duplicates)')
print(f'Columns: {len(raw_df.columns)}')
raw_df.head()

In [ ]:
print('Label distribution (raw, with mixed casing):')
print(raw_df['label_5way'].value_counts())
print('\nChannel distribution (raw, with mixed casing):')
print(raw_df['channel'].value_counts(dropna=False))
print('\nMissing values per column:')
print(raw_df.isna().sum()[raw_df.isna().sum() > 0])

## 1.2 Deduplication

Two passes: exact duplicates first, then near-duplicates (same caller/recipient/minute with duration drift ≤ 5 sec).

In [ ]:
from dataset.dedup import dedup

dedup_df, report = dedup(raw_df)

print(f'Rows before:           {report.rows_before}')
print(f'Exact duplicates:      {report.exact_duplicates}')
print(f'Near-duplicates:       {report.near_duplicates} (across {report.near_dup_groups} groups)')
print(f'Rows after:            {report.rows_after}')

## 1.3 Categorical normalization

The generator deliberately uses mixed casing (`Mobile`/`mobile`/`MOBILE`, `Confirmed_Fraud`/`confirmed_fraud`/`CONFIRMED_FRAUD`). We map everything to a canonical CamelCase form.

In [ ]:
from analysis.cleaning import normalize_categoricals, CHANNEL_MAP, LABEL_MAP

print('Channel BEFORE:', dedup_df['channel'].dropna().unique().tolist())
print('Label BEFORE:', dedup_df['label_5way'].unique().tolist())

cleaned_df = normalize_categoricals(dedup_df)

print('\nChannel AFTER:', cleaned_df['channel'].dropna().unique().tolist())
print('Label AFTER:', cleaned_df['label_5way'].unique().tolist())

(ROOT / 'data').mkdir(exist_ok=True)
cleaned_df.to_csv(ROOT / 'data' / 'cleaned_bank_calls.csv', index=False)
print(f'\nSaved to data/cleaned_bank_calls.csv ({len(cleaned_df)} rows)')

## 1.4 Quality assurance

Six hard checks: no NaN in critical columns, fraud rate in [0.05, 0.25], no negative duration, no duplicate call_ids, valid phone format, all labels in canonical list.

In [ ]:
from dataset.qa import run_checks

checks, _ = run_checks(cleaned_df)
for name, passed, detail in checks:
    status = 'PASS' if passed else 'FAIL'
    print(f'  [{status}] {name}: {detail}')
passed_count = sum(1 for _, p, _ in checks if p)
print(f'\nResult: {passed_count}/{len(checks)} checks passed')

---
# Sprint 2 — Exploratory Data Analysis

## 2.1 Suspicious callers

For every caller number compute: total_calls, distinct_victims, fraud_rate, avg_duration. Filter out callers with < 20 calls to remove single-call noise. Sort by fraud_rate descending.

In [ ]:
from analysis.suspicious_callers import aggregate_by_caller, top_suspicious

agg = aggregate_by_caller(cleaned_df)
top = top_suspicious(agg, n=20, min_calls=20)

print(f'Total unique callers: {len(agg)}')
print(f'Callers with >= 20 calls: {(agg["total_calls"] >= 20).sum()}')
top

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
colors = ['#c0392b' if r > 0.5 else '#e67e22' if r > 0.2 else '#2980b9' for r in top['fraud_rate']]
ax.barh(top.index.astype(str), top['total_calls'], color=colors)
for i, (_, row) in enumerate(top.iterrows()):
    ax.text(row['total_calls'], i, f"  {row['fraud_rate']:.0%} fraud · {int(row['distinct_victims'])} victims", va='center', fontsize=9)
ax.set_xlabel('Total calls')
ax.set_title('Top 20 suspicious caller numbers')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

We see three distinct patterns:
- **Red bars (>50% fraud, high volume)** — mass fraud campaigns
- **Orange (20–50% fraud)** — mixed or targeted attempts
- **Blue (<20% fraud, very high volume)** — banking service numbers (SEB, LHV, Swedbank, etc.)

## 2.2 Complaint keyword analysis

Count occurrences of fraud-indicating keywords in complaint texts grouped by label class. Compute lift = P(keyword | fraud) / P(keyword | other).

In [ ]:
from analysis.complaint_keywords import count_keywords, keyword_correlations

counts = count_keywords(cleaned_df)
print('Per-class keyword frequencies:')
counts

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
counts.T.plot(kind='bar', ax=ax)
ax.set_title('Complaint keyword frequency by class')
ax.set_xlabel('Keyword')
ax.set_ylabel('Frequency')
ax.legend(title='Class', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

Top fraud-signalling keywords: **Smart-ID**, **Mobiil-ID**, **turvakonto** (secure account). These are exactly what real Estonian vishing attackers ask for.

## 2.3 Post-call behavior signals

For each post-call action, compute the conditional probability P(action | class). Fraud classes show 5–10x higher rates of payee creation, transfers, settings changes.

In [ ]:
behavior_cols = ['login_after_call', 'twofa_confirmed_after_call', 'transfer_after_call',
                 'new_payee_after_call', 'settings_changed_after_call']

behavior_by_class = cleaned_df.groupby('label_5way')[behavior_cols].mean()
print('P(action | class):')
behavior_by_class.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
behavior_by_class.T.plot(kind='bar', ax=ax, colormap='RdYlGn_r')
ax.set_title('Post-call action probability by class')
ax.set_ylabel('P(action | class)')
ax.legend(title='Class', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 2.4 Temporal pattern

Correlation between time-to-next-action and fraud label.

In [ ]:
df_tmp = cleaned_df.copy()
df_tmp['is_fraud_binary'] = df_tmp['label_5way'].isin(['Confirmed_Fraud', 'High_Risk_Reported']).astype(int)
corr = df_tmp[['time_to_next_action_min', 'is_fraud_binary']].corr().iloc[0, 1]
print(f'Pearson correlation (time_to_next_action_min vs is_fraud): {corr:.4f}')
print('Negative correlation = faster action after call -> higher fraud probability')

---
# Sprint 3 — Model-Ready Data Prep

## 3.1 Binary target construction

Map 5-class label to binary `is_fraud`. Drop ambiguous classes (`Community_Flagged`, `Unknown`) to avoid diluting the supervised signal.

In [ ]:
from models.targets import build_targets

labeled_df = build_targets(cleaned_df)

print(f'After dropping ambiguous: {len(labeled_df)} rows (was {len(cleaned_df)})')
print(f'Class balance:\n{labeled_df["is_fraud"].value_counts()}')
print(f'Fraud rate: {labeled_df["is_fraud"].mean():.3f}')

labeled_df.to_csv(ROOT / 'data' / 'labeled_bank_calls.csv', index=False)

## 3.2 Feature engineering: temporal + categorical encoding

- Add hour, weekday, is_weekend, is_business_hours from timestamp
- Sort by timestamp ascending (mandatory for time-aware split)
- Encode caller_number_prefix via frequency encoding (target-leak-free)
- One-hot encode channel with drop_first=True
- Drop identifier and text columns

In [ ]:
from models.prepare import add_temporal_features, time_aware_split, numeric_columns_to_scale
from models.encoding import build_features

df = add_temporal_features(labeled_df)
df = df.sort_values('timestamp', kind='mergesort').reset_index(drop=True)
df = build_features(df)
df = df.drop(columns=['confidence_score'])  # leak

print(f'Final feature matrix: {df.shape}')
print(f'Features: {[c for c in df.columns if c != "is_fraud"]}')
df.head()

## 3.3 Time-aware split + StandardScaler

First 80% by time → train, last 20% → test. Scaler fitted on train, applied to test.

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib

train, test = time_aware_split(df, test_size=0.2)

scale_cols = numeric_columns_to_scale(train)
scaler = StandardScaler()
train_scaled = train.copy()
test_scaled = test.copy()
train_scaled[scale_cols] = scaler.fit_transform(train[scale_cols])
test_scaled[scale_cols] = scaler.transform(test[scale_cols])

train.to_csv(ROOT / 'data' / 'train_features.csv', index=False)
test.to_csv(ROOT / 'data' / 'test_features.csv', index=False)
train_scaled.to_csv(ROOT / 'data' / 'train_scaled.csv', index=False)
test_scaled.to_csv(ROOT / 'data' / 'test_scaled.csv', index=False)
joblib.dump(scaler, ROOT / 'models' / 'scaler.joblib')

print(f'Train: {len(train)} rows, fraud rate {train["is_fraud"].mean():.3f}')
print(f'Test:  {len(test)} rows, fraud rate {test["is_fraud"].mean():.3f}')
print(f'Numeric columns scaled: {len(scale_cols)}')

---
# Sprint 4 — Baseline Models

All three models trained on the same train/test split. LR uses scaled features, trees use unscaled. Median imputation fitted on train only.

In [ ]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                             precision_score, recall_score, confusion_matrix,
                             classification_report)

def impute(X_train, X_test):
    medians = X_train.median(numeric_only=True)
    return X_train.fillna(medians), X_test.fillna(medians)

def evaluate(name, model, X_test, y_test):
    start = time.perf_counter()
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    latency_ms = (time.perf_counter() - start) * 1000
    metrics = {
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'f1': f1_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'latency_ms': latency_ms,
    }
    return metrics, confusion_matrix(y_test, y_pred)

X_train_u = train.drop(columns=['is_fraud'])
y_train = train['is_fraud']
X_test_u = test.drop(columns=['is_fraud'])
y_test = test['is_fraud']
X_train_u, X_test_u = impute(X_train_u, X_test_u)

X_train_s = train_scaled.drop(columns=['is_fraud'])
X_test_s = test_scaled.drop(columns=['is_fraud'])
X_train_s, X_test_s = impute(X_train_s, X_test_s)

## 4.1 Logistic Regression (baseline)

In [ ]:
lr = LogisticRegression(max_iter=1000, solver='lbfgs', class_weight='balanced', random_state=42)
lr.fit(X_train_s, y_train)
lr_metrics, lr_cm = evaluate('Logistic Regression', lr, X_test_s, y_test)
print(f"ROC-AUC:   {lr_metrics['roc_auc']:.4f}")
print(f"F1:        {lr_metrics['f1']:.4f}")
print(f"Precision: {lr_metrics['precision']:.4f}")
print(f"Recall:    {lr_metrics['recall']:.4f}")
print(f"Latency:   {lr_metrics['latency_ms']:.2f} ms")
print(f'\nConfusion matrix:\n{lr_cm}')

## 4.2 Random Forest (200 trees, balanced class weights)

In [ ]:
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train_u, y_train)
rf_metrics, rf_cm = evaluate('Random Forest', rf, X_test_u, y_test)
print(f"ROC-AUC:   {rf_metrics['roc_auc']:.4f}")
print(f"F1:        {rf_metrics['f1']:.4f}")
print(f"Precision: {rf_metrics['precision']:.4f}")
print(f"Recall:    {rf_metrics['recall']:.4f}")
print(f"Latency:   {rf_metrics['latency_ms']:.2f} ms")
print(f'\nConfusion matrix:\n{rf_cm}')

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_train_u.columns).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(10, 6))
importances.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Random Forest — Top 15 Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 4.3 Gradient Boosting (default hyperparameters)

In [ ]:
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train_u, y_train)
gb_metrics, gb_cm = evaluate('Gradient Boosting', gb, X_test_u, y_test)
print(f"ROC-AUC:   {gb_metrics['roc_auc']:.4f}")
print(f"F1:        {gb_metrics['f1']:.4f}")
print(f"Precision: {gb_metrics['precision']:.4f}")
print(f"Recall:    {gb_metrics['recall']:.4f}")
print(f"Latency:   {gb_metrics['latency_ms']:.2f} ms")
print(f'\nConfusion matrix:\n{gb_cm}')

## 4.4 Model comparison

In [ ]:
comparison = pd.DataFrame([lr_metrics, rf_metrics, gb_metrics]).set_index('model')
comparison.round(4)

**Winner: Gradient Boosting** — best ROC-AUC, best F1, and 16× faster inference than Random Forest.

---
# Sprint 5 — Optimization

## 5.1 Hyperparameter optimization (GridSearchCV)

27 combinations × 3-fold CV = 81 model fits. Scoring by ROC-AUC.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 150, 200],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [2, 3, 4],
}

search = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid=param_grid,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1,
)
search.fit(X_train_u, y_train)

print(f'Best params: {search.best_params_}')
print(f'Best CV ROC-AUC: {search.best_score_:.4f}')

best_gb = search.best_estimator_
best_metrics, best_cm = evaluate('GB Tuned', best_gb, X_test_u, y_test)
print(f'\nTuned test metrics:')
for k, v in best_metrics.items():
    if k != 'model':
        print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

## 5.2 Behavioral features — the biggest single improvement

Add 4 aggregate features per call:
- `calls_from_number_last_24h` — rolling 24-hour call count for the caller
- `distinct_victims_last_7d` — how many unique people the caller reached in 7 days
- `mean_call_duration_per_number` — caller's average duration
- `complaint_keyword_score` — number of fraud-indicating keywords in complaint text

In [ ]:
from models.behavioral_features import add_aggregate_features, add_temporal_features as add_temp_beh

df_beh = add_temp_beh(labeled_df)
df_beh = add_aggregate_features(df_beh)
df_beh = df_beh.sort_values('timestamp', kind='mergesort').reset_index(drop=True)
df_beh = build_features(df_beh)
if 'confidence_score' in df_beh.columns:
    df_beh = df_beh.drop(columns=['confidence_score'])

train_beh, test_beh = time_aware_split(df_beh)
X_train_beh = train_beh.drop(columns=['is_fraud'])
y_train_beh = train_beh['is_fraud']
X_test_beh = test_beh.drop(columns=['is_fraud'])
y_test_beh = test_beh['is_fraud']
X_train_beh, X_test_beh = impute(X_train_beh, X_test_beh)

print(f'New features: {[c for c in X_train_beh.columns if c not in X_train_u.columns]}')
print(f'Total features now: {X_train_beh.shape[1]}')

## 5.3 Retrain GB with behavioral features and tuned hyperparameters

In [ ]:
final_model = GradientBoostingClassifier(
    learning_rate=0.1,
    max_depth=2,
    n_estimators=200,
    random_state=42,
)
final_model.fit(X_train_beh, y_train_beh)
final_metrics, final_cm = evaluate('Final GB + Behavioral', final_model, X_test_beh, y_test_beh)

print('FINAL MODEL METRICS:')
for k, v in final_metrics.items():
    if k != 'model':
        print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
print(f'\nConfusion matrix:\n{final_cm}')

## 5.4 Improvement table

In [ ]:
improvement = pd.DataFrame({
    'Baseline GB (tuned)': {k: v for k, v in best_metrics.items() if k != 'model'},
    '+ Behavioral features': {k: v for k, v in final_metrics.items() if k != 'model'},
}).T
improvement['delta_recall_pp'] = (improvement['recall'] - improvement['recall'].iloc[0]) * 100
improvement.round(4)

**Key business outcome:** missed fraud cases drop from 18 to 4 — recall jumps +6.25 percentage points.

---
# Sprint 6 — Persist Best Model

Save the trained model + feature column list so the Streamlit dashboard can serve real-time predictions.

In [ ]:
joblib.dump(final_model, ROOT / 'models' / 'best_model.joblib')
joblib.dump(list(X_train_beh.columns), ROOT / 'models' / 'feature_columns.joblib')

print('Saved:')
print('  models/best_model.joblib')
print('  models/feature_columns.joblib')
print('  models/scaler.joblib')
print('\nDashboard now ready. Open http://localhost:8501 (streamlit container).')

---
# Conclusion

| Metric | Logistic Regression | Random Forest | GB Baseline | GB Tuned | **GB + Behavioral** |
|--------|---------------------|---------------|-------------|----------|---------------------|
| ROC-AUC | 0.988 | 0.996 | 0.998 | 0.998 | **0.9999** |
| F1 | 0.901 | 0.952 | 0.954 | 0.947 | **0.987** |
| Recall | 0.933 | 0.933 | 0.933 | 0.920 | **0.982** |
| Latency (ms) | ~3 | ~103 | ~6 | ~6 | ~8 |

**Main insight:** aggregate behavioral features (caller history over 24h / 7d windows) gave a bigger improvement than any algorithm switch or hyperparameter tuning. In fraud detection, *context beats individual signal*.

**Limitations:** synthetic data with embedded patterns produces unrealistically high metrics. Real-world performance would require validation on actual bank data, multilingual NLP, and detection of number spoofing.